<a href="https://colab.research.google.com/github/kanikagoelgupta/AI/blob/main/Edureka_MyCopy_19_May_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install langchain langchain-community langchain-google-genai
!pip -q install pypdf langchain-huggingface langchain-groq sentence-transformers
!pip install langchain-chroma chromadb

In [2]:
import os
from google.colab import userdata
# import getpass
# getpass.getpass("Enter your Groq API key: ")

# Get the Groq API key from Colab secrets
groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key
print("API key is stored")

API key is stored


In [3]:
#Upload the PDF
from google.colab import files
file = files.upload()
filename = list(file.keys())[0]
print("File Uploaded is:", filename)


Saving EmployeeHandbook.pdf to EmployeeHandbook.pdf
File Uploaded is: EmployeeHandbook.pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader

docLoader = PyPDFLoader(filename)
docPages = docLoader.load()

print("Number of pages loaded:", len(docPages))


Number of pages loaded: 60


In [5]:
(docPages[59].page_content[:100])

'GENERAL \n \n \n1.  Employees should contact HR, in case of any clarifications. \n2.  The Company may, n'

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
textSplitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200)
textChunks = textSplitter.split_documents(docPages)
print("Number of chunks:", len(textChunks))
textChunks[115]

Number of chunks: 116


Document(metadata={'producer': 'Acrobat Distiller 5.0 (Windows)', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2013-03-22T11:51:24+05:30', 'moddate': '2013-03-22T11:51:24+05:30', 'author': 'minalk', 'title': 'Microsoft Word - Synise Handbook.doc', 'source': 'EmployeeHandbook.pdf', 'total_pages': 60, 'page': 59, 'page_label': '60'}, page_content='GENERAL \n \n \n1.  Employees should contact HR, in case of any clarifications. \n2.  The Company may, notwithstanding th e eligibility and terms mentioned above, \nat its discretion amend, modify or withdraw this policy. \n3.  Any deviation from the provisions made  in the clauses in the policy will require \nthe prior  approval of  CEO based  on the  recommendation of Head-  Human \nResources. \n \n \n \n \nReviewed by : Ms. Seema Shaikh, Manager HR & Admin. \n \n \n \nApproved by: Mr. Ashok Dani, CEO \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n5

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_ = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

myVectorStore = Chroma.from_documents(
    documents= textChunks,
    embedding=embedding_,
    collection_name ="policy"
)

print("PDF stored in searchable memory successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

PDF stored in searchable memory successfully.


In [8]:
myRetriever = myVectorStore.as_retriever()
print("Retriever is ready")

Retriever is ready


In [21]:
from langchain_groq import ChatGroq

llama_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1
)

print("Groq model is ready.")

Groq model is ready.


In [10]:
from langchain_core.prompts import ChatPromptTemplate

myPrompt = ChatPromptTemplate.from_template("""
You are a helpful Q & A assistant.

Answer the question using only the context provided from the PDF.

If the answer is not available in the PDF context, politely convey that the information is not available at the moment with you.

Keep your answers clear, simple and useful.

Context:
{context}

Question:
{question}

Answer:
""")


In [11]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_doc(docs):
  formatted_text=""
  for doc in docs:
    page_label = doc.metadata.get("page_label", None)
    if not isinstance(page_label, int):
      page_label = "unknown"
    formatted_text += f"Page {page_label}:\n"
    formatted_text += doc.page_content
  return formatted_text

chain = (
    {
        "context": myRetriever | format_doc,
        "question": RunnablePassthrough()
    }
    | myPrompt
    | llama_llm
    | StrOutputParser()
)

In [17]:
question = input("Please enter your question: ")

answer = chain.invoke(question)

print(answer)

Please enter your question: List the key points from this document.
Here are the key points from the document:

1. The document is a personnel manual for Synise Technologies Ltd, introducing employees to the organization's business activities, systems, procedures, and employee benefits.
2. The manual is applicable to all employees in all grades working in Corporate Office and Regional Offices.
3. The manual is available on the Company Intranet and can be referred to by all bonafide employees of Synise Technologies Ltd.
4. The salary offer should be in line with the company's employee levels.
5. An Interview Assessment Sheet is required for all interview remarks of panel members.
6. A background check on the selected candidate is performed through HR before making a final offer.
7. The company has multiple offices in Nasik, Ahmednagar, and Jalgaon, with contact details provided.


In [13]:
def ask_pdf(question):
  answer = chain.invoke(question)
  print(answer)

In [25]:
ask_pdf("List the key points from this document.")

Based on the provided context, here are the key points from the document:

1. The document is a welcome manual for the "Synise" family, introducing employees to the organization's business activities, systems, procedures, working protocol, and employee benefits.
2. The manual is available to all employees in the Corporate Office and Regional Offices.
3. The manual can be referred to by all bonafide employees of Synise Technologies Ltd on the Company Intranet.
4. The manual is for general reference only, and policies and practices may be changed.
5. The HR & Admin Department can be contacted for more information.
6. The manual is applicable to all employees in all grades working in Corporate Office and Regional Offices.
7. A background check is performed on selected candidates before making a final offer.
8. An Interview Assessment Sheet is required for all interview remarks of panel members.
9. The company has multiple offices in Nasik, Ahmednagar, and Jalgaon, with contact details pro

In [26]:
ask_pdf("Summarize this PDF in 5 bullet points.")

Here are 5 bullet points summarizing the PDF:

• The company, Synise, aims to improve its business by transforming its processes into the most efficient and transparent practices using technology, expertise, and cost-effective services.
• The company has an appraisal system that includes objective setting, feedback sharing, and a discussion between the appraiser and appraisee to set achievable objectives.
• The appraisal system is an integral part of the company's performance evaluation and is used to track employee performance and set goals.
• The company has an Activity/Assignment Tracking System to monitor employee progress and objectives.
• The company has a personnel policy handbook that outlines general information, policies, and practices for employees, which is available to all employees in the Corporate Office and Regional Offices.


In [32]:
def ask_pdf_with_sources(question):
  source_docs = myRetriever.invoke(question)

  print("Question:", question)

  print("Retrieved chunks from the document:\n")
  for i,doc in enumerate(source_docs, start=1):
    page_label = doc.metadata.get("page_label", None)
    # if not isinstance(page_label, int):
    #   page_label = "unknown"
    print(f"\n---- Source {i}, Page {page_label} --- \n")
    print(doc.page_content[:700])

  answer = chain.invoke(question)
  print("\nAnswer:", answer)



In [33]:
ask_pdf_with_sources("What are the important rules mentioned about termination in the document?")

Question: What are the important rules mentioned about termination in the document?
Retrieved chunks from the document:


---- Source 1, Page 16 --- 

TERMINATION OF EMPLOYMENT 
 
a)  The service of the employee may be terminated at any time during the tenure of 
his/her employment as follows : 
 
b)  In case of employees who are on probation or working as trainee : By giving one 
(1) month notice (i.e. 30 days notice ) /salary in lieu of notice for ALL employees 
 
c)  In case of confirmed employees : By giving one (1) month notice (i.e. 30 days 
notice )  /  salary in lieu of notice for a ll employees who come under L0, L1, L2A 
grade & for employees who come under  L2 B, L3A, L3B, L4 grade the notice period 
will be two (2) months (i.e. 60 days period) 
 
d)  By the Company without notice, if the employee is found guilty of insubordinat

---- Source 2, Page 19 --- 

CLEARANCE FORMALITIES 
 
In the event of termination of the employment in terms of resignation, the 
following procedu